# Donut Template Projection Training

Same idea as `train_model_practice.ipynb`, but with a Donut/Swin visual backbone. Donut has no LayoutLM-style CLS token here, so the visual token sequence is average pooled before the projection head.

In [1]:
import os
os.environ.setdefault("TRANSFORMERS_NO_TF", "1")
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import csv
import json
import math
import random
import time
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as func
from torch.utils.data import BatchSampler, DataLoader, Dataset
from PIL import Image
import matplotlib.pyplot as plt

from transformers import DonutImageProcessor, VisionEncoderDecoderModel, get_linear_schedule_with_warmup

In [ ]:
SPLIT_OCR_ROOT = Path(r"A:\RealForm\processed\ocr_cache_splitted")
DEGRADED_ROOT = Path(r"A:\RealForm\processed\synthetic_fill_degraded")
ORIGINAL_ROOT = Path(r"A:\RealForm\processed\CommonFormsEnglish\images")
OUTPUT_DIR = Path(r"A:\RealForm\outputs\practice_train_donut")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DONUT_MODEL_NAME = "naver-clova-ix/donut-base"
IMAGE_SIZE = {"height": 960, "width": 720}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

device: cuda


In [3]:
def read_json(path):
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)

@dataclass(frozen=True)
class DocSample:
    subset: str
    stem: str
    class_label: str
    image_path: Path
    record_id: int
    is_original: bool = False

In [4]:
def build_original_image_index(original_root):
    index = {}
    for image_path in sorted(original_root.rglob("*"), key=lambda p: str(p).lower()):
        if image_path.is_file() and image_path.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp", ".tif", ".tiff"}:
            index.setdefault(image_path.stem, image_path)
    return index


def load_split(split_name):
    original_index = build_original_image_index(ORIGINAL_ROOT)
    query_docs = []
    original_docs = []

    split_dir = SPLIT_OCR_ROOT / split_name
    template_dirs = sorted([p for p in split_dir.iterdir() if p.is_dir()], key=lambda p: p.name.lower())

    for template_dir in template_dirs:
        stem = template_dir.name
        original_image = original_index.get(stem)
        degraded_dir = DEGRADED_ROOT / stem

        if original_image is None or not degraded_dir.exists():
            continue

        original_docs.append(
            DocSample(
                subset="original",
                stem=stem,
                class_label=stem,
                image_path=original_image.resolve(),
                record_id=0,
                is_original=True,
            )
        )

        # Keep the same split membership as the OCR split. Donut does not use OCR, but the OCR files tell us which fills belong here.
        for record_id, image_path in enumerate(sorted(degraded_dir.glob("fill_*.png"), key=lambda p: p.name.lower()), start=1):
            ocr_path = template_dir / f"{image_path.stem}.ocr.json"
            if not ocr_path.exists():
                continue
            query_docs.append(
                DocSample(
                    subset=split_name,
                    stem=stem,
                    class_label=stem,
                    image_path=image_path.resolve(),
                    record_id=record_id,
                    is_original=False,
                )
            )

    return query_docs, original_docs


train_query_docs, train_original_docs = load_split("train")
val_query_docs, val_original_docs = load_split("val")
test_query_docs, test_original_docs = load_split("test")

print("train", len(train_original_docs), len(train_query_docs))
print("val", len(val_original_docs), len(val_query_docs))
print("test", len(test_original_docs), len(test_query_docs))

train 8779 87790
val 285 2850
test 279 2790


In [5]:
class DenseRetriveHead(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size // 2, 128),
        )

    def forward(self, pooled):
        return func.normalize(self.net(pooled), p=2, dim=-1)


class AccelerateArcFace(nn.Module):
    def __init__(self, emb_dim, num_class):
        super().__init__()
        self.param = nn.Parameter(torch.empty(num_class, emb_dim))
        nn.init.xavier_uniform_(self.param)
        self.cos_margin = math.cos(0.2)
        self.sin_margin = math.sin(0.2)

    def forward(self, embed, labels):
        norm_emb = func.normalize(embed, p=2, dim=-1)
        norm_weight = func.normalize(self.param, p=2, dim=-1)
        cos = func.linear(norm_emb, norm_weight).clamp(-1.0 + 1e-7, 1.0 - 1e-7)
        sin = torch.sqrt(torch.clamp(1.0 - cos * cos, min=1e-7))
        phi = cos * self.cos_margin - sin * self.sin_margin
        one_hot = func.one_hot(labels, num_classes=self.param.shape[0]).to(dtype=cos.dtype)
        return (one_hot * phi + (1.0 - one_hot) * cos) * 30.0


def sup_con_loss(embs, labels, temperature=0.1):
    norm = func.normalize(embs, p=2, dim=-1)
    logits = norm @ norm.T / temperature
    logits = logits - logits.max(dim=1, keepdim=True).values.detach()
    self_mask = torch.eye(labels.shape[0], device=labels.device, dtype=torch.bool)
    positive_mask = labels.unsqueeze(0).eq(labels.unsqueeze(1)) & ~self_mask
    exp_logits = torch.exp(logits) * (~self_mask)
    log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True).clamp(min=1e-12))
    positive_counts = positive_mask.sum(dim=1)
    mean_log_prob = (positive_mask * log_prob).sum(dim=1) / positive_counts.clamp(min=1)
    valid = positive_counts > 0
    return embs.new_tensor(0.0) if not torch.any(valid) else -mean_log_prob[valid].mean()

In [6]:
def find_donut_blocks(backbone):
    blocks = []
    encoder = getattr(backbone, "encoder", None)
    stages = getattr(encoder, "layers", []) if encoder is not None else []
    for stage in stages:
        if hasattr(stage, "blocks"):
            blocks.extend(list(stage.blocks))
        elif hasattr(stage, "layers"):
            blocks.extend(list(stage.layers))
        else:
            blocks.append(stage)
    return blocks


class DonutODCModel(nn.Module):
    def __init__(self, num_class):
        super().__init__()
        self.backbone = VisionEncoderDecoderModel.from_pretrained(
            DONUT_MODEL_NAME,
            use_safetensors=True,
        ).encoder
        self.backbone.gradient_checkpointing_enable()
        hidden_size = self.backbone.config.hidden_size
        self.proj = DenseRetriveHead(hidden_size)
        self.accelerate = AccelerateArcFace(128, num_class)
        self.unfreeze_last_4_layers()
        self.optimizer = self.create_optimizer()

    def forward(self, pixel_values):
        outputs = self.backbone(pixel_values=pixel_values, return_dict=True)
        # Donut/Swin gives visual tokens, not a LayoutLM CLS token. Average pool over tokens.
        pooled = outputs.last_hidden_state.mean(dim=1)
        return self.proj(pooled)

    def unfreeze_last_4_layers(self):
        for param in self.backbone.parameters():
            param.requires_grad = False

        blocks = find_donut_blocks(self.backbone)
        if blocks:
            for block in blocks[-4:]:
                for param in block.parameters():
                    param.requires_grad = True
        else:
            print("WARNING: could not find Donut/Swin blocks; only projection head will train")

        # Final normalization can matter when only the tail blocks are trainable.
        for name, module in self.backbone.named_modules():
            if "layernorm" in name.lower() or name.lower().endswith("norm"):
                for param in module.parameters(recurse=False):
                    param.requires_grad = True

        for param in self.proj.parameters():
            param.requires_grad = True
        for param in self.accelerate.parameters():
            param.requires_grad = True

        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        print(f"trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")
        print(f"donut blocks found: {len(blocks)}; unfrozen last: {min(4, len(blocks))}")

    def create_optimizer(self):
        back_param = [p for p in self.backbone.parameters() if p.requires_grad]
        proj_param = [p for p in self.proj.parameters() if p.requires_grad]
        acc_param = [p for p in self.accelerate.parameters() if p.requires_grad]
        return torch.optim.AdamW(
            [
                {"params": back_param, "lr": 2e-5, "weight_decay": 0.01},
                {"params": proj_param, "lr": 1e-4, "weight_decay": 0.01},
                {"params": acc_param, "lr": 1e-4, "weight_decay": 0.01},
            ]
        )

In [7]:
@dataclass(frozen=True)
class TemplateExample:
    document: DocSample
    class_label: str
    train_label_id: int


class TemplateDocumentDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


class DonutCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, examples):
        images = [Image.open(ex.document.image_path).convert("RGB") for ex in examples]
        try:
            encoding = self.processor(images=images, return_tensors="pt")
        finally:
            for img in images:
                img.close()
        return {
            "pixel_values": encoding["pixel_values"],
            "labels": torch.tensor([ex.train_label_id for ex in examples], dtype=torch.long),
            "documents": [ex.document for ex in examples],
        }


processor = DonutImageProcessor.from_pretrained(DONUT_MODEL_NAME, size=IMAGE_SIZE)
collator = DonutCollator(processor)

In [8]:
class BalanceBatchSampler(BatchSampler):
    def __init__(self, examples, classes_per_batch=2, samples_per_class=1, batches_per_epoch=None, seed=42):
        self.examples = examples
        self.classes_per_batch = classes_per_batch
        self.samples_per_class = samples_per_class
        self.seed = seed
        self.class_to_indices = {}
        for idx, ex in enumerate(examples):
            self.class_to_indices.setdefault(ex.train_label_id, []).append(idx)
        self.class_ids = sorted(self.class_to_indices.keys())
        default_batches = math.ceil(len(examples) / max(1, classes_per_batch * samples_per_class))
        self.batches_per_epoch = batches_per_epoch or default_batches

    def __len__(self):
        return self.batches_per_epoch

    def __iter__(self):
        rng = random.Random(self.seed + int(time.time()))
        for _ in range(self.batches_per_epoch):
            chosen_classes = rng.sample(self.class_ids, k=min(self.classes_per_batch, len(self.class_ids)))
            batch_indices = []
            for class_id in chosen_classes:
                candidates = self.class_to_indices[class_id]
                if len(candidates) >= self.samples_per_class:
                    batch_indices.extend(rng.sample(candidates, k=self.samples_per_class))
                else:
                    batch_indices.extend(rng.choices(candidates, k=self.samples_per_class))
            yield batch_indices


def build_examples(documents, class_to_id):
    return [
        TemplateExample(document=doc, class_label=doc.class_label, train_label_id=class_to_id[doc.class_label])
        for doc in documents
    ]


def build_inference_examples(documents):
    labels = sorted({doc.class_label for doc in documents})
    class_to_id = {label: idx for idx, label in enumerate(labels)}
    return build_examples(documents, class_to_id)


train_labels = sorted({doc.class_label for doc in train_query_docs})
train_class_to_id = {label: idx for idx, label in enumerate(train_labels)}
train_examples = build_examples(train_query_docs, train_class_to_id)
train_dataset = TemplateDocumentDataset(train_examples)

train_sampler = BalanceBatchSampler(
    examples=train_examples,
    classes_per_batch=2,
    samples_per_class=2,
    batches_per_epoch=None,
    seed=42,
)

train_loader = DataLoader(
    train_dataset,
    batch_sampler=train_sampler,
    num_workers=0,
    collate_fn=collator,
)

print("train labels/templates:", len(train_labels))
print("train examples:", len(train_examples))
print("train batches:", len(train_loader))

train labels/templates: 8779
train examples: 87790
train batches: 21948


In [9]:
EPOCHS = 10
model = DonutODCModel(num_class=len(train_labels)).to(DEVICE)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * 0.1)

scheduler = get_linear_schedule_with_warmup(
    model.optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")

print("total_steps:", total_steps)
print("warmup_steps:", warmup_steps)

trainable params: 33,283,168 / 75,896,952 (43.85%)
donut blocks found: 20; unfrozen last: 4


C:\Users\thanh\anaconda3\envs\DonutTorch26\lib\site-packages\torch\nn\modules\module.py:1329: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(


total_steps: 219480
warmup_steps: 21948


In [ ]:
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    total_arc = 0.0
    total_supcon = 0.0

    for step, batch in enumerate(train_loader, start=1):
        pixel_values = batch["pixel_values"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        model.optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            embs = model(pixel_values=pixel_values)
            arc_logits = model.accelerate(embs, labels)
            arc_loss = func.cross_entropy(arc_logits, labels)
            contrast_loss = sup_con_loss(embs, labels)
            loss = 0.2 * arc_loss + 0.8 * contrast_loss

        scaler.scale(loss).backward()
        scaler.unscale_(model.optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(model.optimizer)
        scaler.update()
        scheduler.step()

        total_loss += float(loss.item())
        total_arc += float(arc_loss.item())
        total_supcon += float(contrast_loss.item())

        if step == 1 or step % 20 == 0 or step == len(train_loader):
            print(
                f"epoch {epoch} step {step}/{len(train_loader)} "
                f"loss={loss.item():.4f} arc={arc_loss.item():.4f} supcon={contrast_loss.item():.4f}"
            )

    row = {
        "epoch": epoch,
        "train_loss": total_loss / len(train_loader),
        "train_arc_loss": total_arc / len(train_loader),
        "train_supcon_loss": total_supcon / len(train_loader),
    }
    history.append(row)
    print("epoch complete:", row)

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": model.optimizer.state_dict(),
            "history": history,
            "model_name": DONUT_MODEL_NAME,
            "pooling": "mean_pool_last_hidden_state",
        },
        OUTPUT_DIR / "last_donut_projection_model.pt",
    )

epoch 1 step 1/21948 loss=4.1817 arc=16.9180 supcon=0.9976
epoch 1 step 20/21948 loss=4.4510 arc=18.2734 supcon=0.9954
epoch 1 step 40/21948 loss=5.4599 arc=23.0781 supcon=1.0554
epoch 1 step 60/21948 loss=4.4055 arc=17.9648 supcon=1.0157


In [ ]:
import gc, torch
gc.collect() # Reclaims Python memory
torch.cuda.empty_cache() # Releases unused cached GPU memory
torch.cuda.empty_cache()

In [ ]:
def build_inference_loader(query_docs, original_docs, batch_size=2):
    docs = query_docs + original_docs
    examples = build_inference_examples(docs)
    return DataLoader(
        TemplateDocumentDataset(examples),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        collate_fn=collator,
    )


def collect_embeddings(model, loader):
    model.eval()
    all_embs = []
    all_docs = []
    with torch.no_grad():
        for batch in loader:
            pixel_values = batch["pixel_values"].to(DEVICE)
            embs = model(pixel_values=pixel_values)
            all_embs.append(embs.detach().cpu().numpy())
            all_docs.extend(batch["documents"])
    return np.concatenate(all_embs, axis=0), all_docs


def retrieval_vs_original(query_docs, query_embs, original_docs, original_embs):
    query_embs = query_embs / np.maximum(np.linalg.norm(query_embs, axis=1, keepdims=True), 1e-12)
    original_embs = original_embs / np.maximum(np.linalg.norm(original_embs, axis=1, keepdims=True), 1e-12)
    original_labels = np.array([doc.stem for doc in original_docs])

    top1 = top5 = top10 = 0
    reciprocal_ranks = []
    for i, query_doc in enumerate(query_docs):
        scores = query_embs[i] @ original_embs.T
        order = np.argsort(-scores)
        ranked_labels = original_labels[order]
        rank = np.where(ranked_labels == query_doc.stem)[0][0] + 1
        top1 += int(rank == 1)
        top5 += int(rank <= 5)
        top10 += int(rank <= 10)
        reciprocal_ranks.append(1.0 / rank)

    n = len(query_docs)
    return {
        "query_count": n,
        "gallery_count": len(original_docs),
        "retrieval_at_1": top1 / n,
        "retrieval_at_5": top5 / n,
        "retrieval_at_10": top10 / n,
        "mrr": float(sum(reciprocal_ranks) / n),
        "correct_top1": top1,
    }


def evaluate_split(query_docs, original_docs, name):
    loader = build_inference_loader(query_docs, original_docs, batch_size=2)
    embs, docs = collect_embeddings(model, loader)
    query_count = len(query_docs)
    original_count = len(original_docs)
    query_embs = embs[:query_count]
    original_embs = embs[query_count:query_count + original_count]
    metrics = retrieval_vs_original(query_docs, query_embs, original_docs, original_embs)
    print(name, metrics)
    write_json(OUTPUT_DIR / f"{name}_retrieval_metrics.json", metrics)
    return metrics

In [ ]:
val_metrics = evaluate_split(
    query_docs=val_query_docs,
    original_docs=val_original_docs,
    name="val",
)

In [ ]:
test_metrics = evaluate_split(
    query_docs=test_query_docs,
    original_docs=test_original_docs,
    name="test",
)

In [ ]:
FINAL_MODEL_PATH = OUTPUT_DIR / "donut_projection_scripted.pt"

class DonutModelWrap(nn.Module):
    def __init__(self, trained_model):
        super().__init__()
        self.backbone = trained_model.backbone
        self.proj = trained_model.proj

    def forward(self, pixel_values):
        outputs = self.backbone(pixel_values=pixel_values, return_dict=True)
        pooled = outputs.last_hidden_state.mean(dim=1)
        return self.proj(pooled)


model.eval()
model.cpu()
wrapped_model = DonutModelWrap(model).eval().cpu()
example_pixel_values = torch.zeros((1, 3, IMAGE_SIZE["height"], IMAGE_SIZE["width"]), dtype=torch.float32)

with torch.no_grad():
    traced = torch.jit.trace(wrapped_model, (example_pixel_values,), strict=False)

FINAL_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
traced.save(str(FINAL_MODEL_PATH))
print("saved", FINAL_MODEL_PATH)